# **E-commerce Discount Framing A/B Test: ₹200 OFF vs 20% OFF**

**Note:** This is a synthetic dataset created for portfolio practice. It does not represent any real company or real experiment.

## Objective
Test whether presenting the same ₹200 saving as "₹200 OFF" (Group A, Control) or "20% OFF" (Group B, Treatment) changes purchase conversion on a ₹1,000 cart.

## Metrics:
- **Primary:** Purchase Conversion Rate
- **Guardrails:** Complaint Rate, Return Rate (among purchasers)

## Phase 1: Data Understanding & Validation
### 1.1 Load Dataset and Inspect Structure

In [1]:
import pandas as pd

df = pd.read_csv("ecommerce_discount_ab_test.csv")

print("Shape (rows, columns):", df.shape)
print()
print("Column names:", list(df.columns))
print()
print(df.dtypes)
print()
print(df.head(10))

Shape (rows, columns): (20000, 8)

Column names: ['User_ID', 'Group', 'Offer', 'Cart_Value', 'Purchased', 'Order_Value', 'Complaint', 'Returned']

User_ID        object
Group          object
Offer          object
Cart_Value      int64
Purchased       int64
Order_Value     int64
Complaint       int64
Returned        int64
dtype: object

  User_ID Group     Offer  Cart_Value  Purchased  Order_Value  Complaint  \
0  U00001     B   20% OFF        1000          0            0          0   
1  U00002     B   20% OFF        1000          0            0          0   
2  U00003     B   20% OFF        1000          0            0          0   
3  U00004     B   20% OFF        1000          0            0          0   
4  U00005     B   20% OFF        1000          0            0          0   
5  U00006     B   20% OFF        1000          0            0          0   
6  U00007     A  ₹200 OFF        1000          0            0          0   
7  U00008     A  ₹200 OFF        1000          0      

### 1.2 Check Missing Values and Duplicate Users

In [2]:
print("Missing values per column:")
print(df.isnull().sum())
print()

print("Total duplicate rows:", df.duplicated().sum())
print("Duplicate User_IDs:", df["User_ID"].duplicated().sum())
print("Unique User_IDs:", df["User_ID"].nunique())

Missing values per column:
User_ID        0
Group          0
Offer          0
Cart_Value     0
Purchased      0
Order_Value    0
Complaint      0
Returned       0
dtype: int64

Total duplicate rows: 0
Duplicate User_IDs: 0
Unique User_IDs: 20000


### 1.3 Validate Group Distribution and Offer Mapping

In [3]:
print("Users per group:")
print(df["Group"].value_counts())
print()

print("Group vs Offer cross-tab:")
print(pd.crosstab(df["Group"], df["Offer"]))
print()

print("Unique Cart_Value:", df["Cart_Value"].unique())

Users per group:
Group
B    10000
A    10000
Name: count, dtype: int64

Group vs Offer cross-tab:
Offer  20% OFF  ₹200 OFF
Group                   
A            0     10000
B        10000         0

Unique Cart_Value: [1000]


### 1.4 Validate Business Logic Rules


In [4]:
rules = {
    "Rule 1: Purchased=0 -> Order_Value=0":
        ((df["Purchased"] == 0) & (df["Order_Value"] != 0)).sum(),
    "Rule 2: Purchased=1 -> Order_Value=800":
        ((df["Purchased"] == 1) & (df["Order_Value"] != 800)).sum(),
    "Rule 3: Purchased=0 -> Complaint=0":
        ((df["Purchased"] == 0) & (df["Complaint"] != 0)).sum(),
    "Rule 4: Purchased=0 -> Returned=0":
        ((df["Purchased"] == 0) & (df["Returned"] != 0)).sum(),
    "Rule 5: Complaint=1 -> Purchased=1":
        ((df["Complaint"] == 1) & (df["Purchased"] != 1)).sum(),
    "Rule 6: Returned=1 -> Purchased=1":
        ((df["Returned"] == 1) & (df["Purchased"] != 1)).sum(),
}

validation = pd.DataFrame({
    "Violations": pd.Series(rules)
})
validation["Status"] = validation["Violations"].apply(lambda x: "PASS" if x == 0 else "FAIL")

print(validation)
print()

for col in ["Purchased", "Complaint", "Returned"]:
    print(col, "unique values:", sorted(df[col].unique()))

                                        Violations Status
Rule 1: Purchased=0 -> Order_Value=0             0   PASS
Rule 2: Purchased=1 -> Order_Value=800           0   PASS
Rule 3: Purchased=0 -> Complaint=0               0   PASS
Rule 4: Purchased=0 -> Returned=0                0   PASS
Rule 5: Complaint=1 -> Purchased=1               0   PASS
Rule 6: Returned=1 -> Purchased=1                0   PASS

Purchased unique values: [np.int64(0), np.int64(1)]
Complaint unique values: [np.int64(0), np.int64(1)]
Returned unique values: [np.int64(0), np.int64(1)]


## Phase 2: SQL Analysis
### 2.0 Create SQLite Database and Load Data
The validated dataset is loaded into a SQLite table named `ab_test` so that all experiment metrics can be calculated using SQL.

In [5]:
import sqlite3

conn = sqlite3.connect("ab_test.db")
df.to_sql("ab_test", conn, if_exists="replace", index=False)

print(pd.read_sql("SELECT * FROM ab_test LIMIT 5;", conn))

  User_ID Group    Offer  Cart_Value  Purchased  Order_Value  Complaint  \
0  U00001     B  20% OFF        1000          0            0          0   
1  U00002     B  20% OFF        1000          0            0          0   
2  U00003     B  20% OFF        1000          0            0          0   
3  U00004     B  20% OFF        1000          0            0          0   
4  U00005     B  20% OFF        1000          0            0          0   

   Returned  
0         0  
1         0  
2         0  
3         0  
4         0  


### 2.1 Total Users in the Experiment

In [6]:
query = """
SELECT
    COUNT(*) AS total_users,
    COUNT(DISTINCT User_ID) AS unique_users
FROM ab_test;
"""

pd.read_sql(query, conn)

,total_users,unique_users
0,20000,20000


### 2.2 Users and Purchases by Group

In [7]:
query = """
SELECT
    "Group",
    Offer,
    COUNT(*) AS users,
    SUM(Purchased) AS purchases
FROM ab_test
GROUP BY "Group", Offer
ORDER BY "Group";
"""

pd.read_sql(query, conn)

,Group,Offer,users,purchases
0,A,₹200 OFF,10000,819
1,B,20% OFF,10000,980


### 2.3 Purchase Conversion Rate by Group
Conversion Rate = Purchases / Users x 100

In [8]:
query = """
SELECT
    "Group",
    Offer,
    COUNT(*) AS users,
    SUM(Purchased) AS purchases,
    ROUND(100.0 * SUM(Purchased) / COUNT(*), 2) AS conversion_rate_pct
FROM ab_test
GROUP BY "Group", Offer
ORDER BY "Group";
"""

pd.read_sql(query, conn)

,Group,Offer,users,purchases,conversion_rate_pct
0,A,₹200 OFF,10000,819,8.19
1,B,20% OFF,10000,980,9.80


### 2.4 Absolute Difference in Conversion Rate
Absolute Difference = Conversion Rate (B) - Conversion Rate (A), measured in percentage points (pp)

In [9]:
query = """
WITH rates AS (
    SELECT
        "Group" AS grp,
        100.0 * SUM(Purchased) / COUNT(*) AS conv
    FROM ab_test
    GROUP BY "Group"
)
SELECT
    ROUND(MAX(CASE WHEN grp = 'A' THEN conv END), 2) AS conv_A_pct,
    ROUND(MAX(CASE WHEN grp = 'B' THEN conv END), 2) AS conv_B_pct,
    ROUND(
        MAX(CASE WHEN grp = 'B' THEN conv END)
        - MAX(CASE WHEN grp = 'A' THEN conv END), 2
    ) AS absolute_diff_pp
FROM rates;
"""

pd.read_sql(query, conn)

,conv_A_pct,conv_B_pct,absolute_diff_pp
0,8.19,9.8,1.61


### 2.5 Relative Lift
Relative Lift = (Conversion B - Conversion A) / Conversion A x 100

In [10]:
query = """
WITH rates AS (
    SELECT
        "Group" AS grp,
        100.0 * SUM(Purchased) / COUNT(*) AS conv
    FROM ab_test
    GROUP BY "Group"
),
pivoted AS (
    SELECT
        MAX(CASE WHEN grp = 'A' THEN conv END) AS conv_A,
        MAX(CASE WHEN grp = 'B' THEN conv END) AS conv_B
    FROM rates
)
SELECT
    ROUND(conv_A, 2) AS conv_A_pct,
    ROUND(conv_B, 2) AS conv_B_pct,
    ROUND(conv_B - conv_A, 2) AS absolute_diff_pp,
    ROUND(100.0 * (conv_B - conv_A) / conv_A, 2) AS relative_lift_pct
FROM pivoted;
"""

pd.read_sql(query, conn)

,conv_A_pct,conv_B_pct,absolute_diff_pp,relative_lift_pct
0,8.19,9.8,1.61,19.66


### 2.6 Complaint Rate (Guardrail Metric)
Complaint Rate = Complaints / Purchasers x 100 (calculated among purchasers only)

In [11]:
query = """
SELECT
    "Group",
    Offer,
    SUM(Purchased) AS purchasers,
    SUM(Complaint) AS complaints,
    ROUND(100.0 * SUM(Complaint) / SUM(Purchased), 2) AS complaint_rate_pct
FROM ab_test
GROUP BY "Group", Offer
ORDER BY "Group";
"""

pd.read_sql(query, conn)

,Group,Offer,purchasers,complaints,complaint_rate_pct
0,A,₹200 OFF,819,20,2.44
1,B,20% OFF,980,26,2.65


### 2.7 Return Rate (Guardrail Metric)
Return Rate = Returned Orders / Purchasers x 100 (calculated among purchasers only)

In [12]:
query = """
SELECT
    "Group",
    Offer,
    SUM(Purchased) AS purchasers,
    SUM(Returned) AS returns,
    ROUND(100.0 * SUM(Returned) / SUM(Purchased), 2) AS return_rate_pct
FROM ab_test
GROUP BY "Group", Offer
ORDER BY "Group";
"""

pd.read_sql(query, conn)

,Group,Offer,purchasers,returns,return_rate_pct
0,A,₹200 OFF,819,45,5.49
1,B,20% OFF,980,52,5.31


### 2.8 Orders, Total Order Value and AOV (Sanity Check)
AOV = Total Order Value / Number of Orders. Every completed order is designed to be ₹800 (₹1,000 cart minus ₹200 discount), so AOV is a diagnostic check rather than a performance metric.

In [13]:
query = """
SELECT
    "Group",
    Offer,
    SUM(Purchased) AS orders,
    SUM(Order_Value) AS total_order_value,
    ROUND(1.0 * SUM(Order_Value) / SUM(Purchased), 2) AS aov
FROM ab_test
GROUP BY "Group", Offer
ORDER BY "Group";
"""

pd.read_sql(query, conn)

,Group,Offer,orders,total_order_value,aov
0,A,₹200 OFF,819,655200,800.0
1,B,20% OFF,980,784000,800.0


### 2.9 Final Experiment Summary
All experiment metrics in one table, by group.

In [14]:
query = """
SELECT
    "Group" AS grp,
    Offer AS offer,
    COUNT(*) AS users,
    SUM(Purchased) AS purchasers,
    ROUND(100.0 * SUM(Purchased) / COUNT(*), 2) AS conversion_pct,
    SUM(Complaint) AS complaints,
    ROUND(100.0 * SUM(Complaint) / SUM(Purchased), 2) AS complaint_rate_pct,
    SUM(Returned) AS returns,
    ROUND(100.0 * SUM(Returned) / SUM(Purchased), 2) AS return_rate_pct,
    SUM(Order_Value) AS total_order_value,
    ROUND(1.0 * SUM(Order_Value) / SUM(Purchased), 2) AS aov
FROM ab_test
GROUP BY "Group", Offer
ORDER BY "Group";
"""

summary = pd.read_sql(query, conn)
summary.set_index("grp").T

grp,A,B
offer,₹200 OFF,20% OFF
users,10000,10000
purchasers,819,980
conversion_pct,8.19,9.8
complaints,20,26
complaint_rate_pct,2.44,2.65
returns,45,52
return_rate_pct,5.49,5.31
total_order_value,655200,784000
aov,800.0,800.0


## Phase 3: Python Statistical Analysis
### 3.1 Conversion Rate by Group (Python Cross-Check)

In [15]:
summary_py = df.groupby("Group").agg(
    users=("User_ID", "count"),
    purchasers=("Purchased", "sum")
)
summary_py["conversion_pct"] = (summary_py["purchasers"] / summary_py["users"] * 100).round(2)

summary_py

,users,purchasers,conversion_pct
Group,,,
A,10000,819,8.19
B,10000,980,9.80


### 3.2 Absolute Difference and Relative Lift


In [16]:
# Group totals (x = purchasers, n = users)
x_A = int(summary_py.loc["A", "purchasers"])
n_A = int(summary_py.loc["A", "users"])
x_B = int(summary_py.loc["B", "purchasers"])
n_B = int(summary_py.loc["B", "users"])

# Conversion rates as proportions
p_A = x_A / n_A
p_B = x_B / n_B

# Effect size
abs_diff_pp = (p_B - p_A) * 100
relative_lift_pct = (p_B - p_A) / p_A * 100

print(f"Group A: {x_A} / {n_A} = {p_A*100:.2f}%")
print(f"Group B: {x_B} / {n_B} = {p_B*100:.2f}%")
print(f"Absolute difference: {abs_diff_pp:.2f} percentage points")
print(f"Relative lift: {relative_lift_pct:.2f}%")

Group A: 819 / 10000 = 8.19%
Group B: 980 / 10000 = 9.80%
Absolute difference: 1.61 percentage points
Relative lift: 19.66%


### 3.3 Hypotheses and Test Selection

**Primary metric:** Purchase Conversion Rate

- **H0 (Null):** Conversion rate of Group A (₹200 OFF) = Conversion rate of Group B (20% OFF)
- **H1 (Alternative):** Conversion rate of Group A ≠ Conversion rate of Group B (two-sided)
- **Significance level:** α = 0.05
- **Test:** Two-proportion z-test

**Why this test:** The outcome is binary (purchased or not), the two groups are independent, and the sample is large (10,000 users per group).

### 3.4 Two-Proportion Z-Test (Primary Metric)

In [17]:
from statsmodels.stats.proportion import proportions_ztest

count = [x_B, x_A]   # purchasers in B, then A
nobs = [n_B, n_A]    # users in B, then A

z_stat, p_value = proportions_ztest(count, nobs, alternative="two-sided")

print(f"Z-statistic: {z_stat:.4f}")
print(f"P-value: {p_value:.6f}")
print(f"P-value (scientific): {p_value:.3e}")
print()
print("Decision at alpha = 0.05:",
      "Reject H0" if p_value < 0.05 else "Fail to reject H0")

Z-statistic: 3.9790
P-value: 0.000069
P-value (scientific): 6.920e-05

Decision at alpha = 0.05: Reject H0


### 3.5 95% Confidence Interval for the Difference in Conversion
CI = (p_B - p_A) ± 1.96 × SE, where SE = sqrt( p_A(1-p_A)/n_A + p_B(1-p_B)/n_B )

In [18]:
import numpy as np

diff = p_B - p_A
se = np.sqrt(p_A * (1 - p_A) / n_A + p_B * (1 - p_B) / n_B)
margin = 1.96 * se

ci_low = (diff - margin) * 100
ci_high = (diff + margin) * 100

print(f"Observed difference: {diff*100:.2f} percentage points")
print(f"Standard error: {se*100:.3f} percentage points")
print(f"95% CI for difference (B - A): [{ci_low:.2f}, {ci_high:.2f}] percentage points")
print("Does the CI include 0?", "Yes" if ci_low <= 0 <= ci_high else "No")

Observed difference: 1.61 percentage points
Standard error: 0.404 percentage points
95% CI for difference (B - A): [0.82, 2.40] percentage points
Does the CI include 0? No


### 3.6 Practical Significance: Estimated Business Impact
Translating the conversion difference into extra purchases and extra order value per 10,000 users exposed. Order value is ₹800 per purchase (already net of the ₹200 discount).

In [19]:
users_scale = 10000
order_value = 800

for label, d in [("Lower CI bound", ci_low),
                 ("Observed difference", diff * 100),
                 ("Upper CI bound", ci_high)]:
    extra_orders = d / 100 * users_scale
    extra_value = extra_orders * order_value
    print(f"{label}: {d:.2f} pp -> ~{extra_orders:.0f} extra purchases "
          f"-> ~Rs {extra_value:,.0f} extra order value per {users_scale:,} users")

Lower CI bound: 0.82 pp -> ~82 extra purchases -> ~Rs 65,381 extra order value per 10,000 users
Observed difference: 1.61 pp -> ~161 extra purchases -> ~Rs 128,800 extra order value per 10,000 users
Upper CI bound: 2.40 pp -> ~240 extra purchases -> ~Rs 192,219 extra order value per 10,000 users


### 3.7 Guardrail 1: Complaint Rate (among purchasers)
- **H0:** Complaint rate of A = Complaint rate of B
- **H1:** Complaint rate of A ≠ Complaint rate of B
- **Denominator:** purchasers only (819 in A, 980 in B)
- **Test:** Two-proportion z-test, alpha = 0.05

In [20]:
# Complaints and purchasers per group
c_A = int(df[df["Group"] == "A"]["Complaint"].sum())
c_B = int(df[df["Group"] == "B"]["Complaint"].sum())
pur_A = x_A   # purchasers in A
pur_B = x_B   # purchasers in B

cr_A = c_A / pur_A
cr_B = c_B / pur_B

# Z-test
z_c, p_c = proportions_ztest([c_B, c_A], [pur_B, pur_A], alternative="two-sided")

# 95% CI for the difference (B - A)
diff_c = cr_B - cr_A
se_c = np.sqrt(cr_A * (1 - cr_A) / pur_A + cr_B * (1 - cr_B) / pur_B)
ci_c_low = (diff_c - 1.96 * se_c) * 100
ci_c_high = (diff_c + 1.96 * se_c) * 100

print(f"Complaint rate A: {c_A}/{pur_A} = {cr_A*100:.2f}%")
print(f"Complaint rate B: {c_B}/{pur_B} = {cr_B*100:.2f}%")
print(f"Difference (B - A): {diff_c*100:.2f} percentage points")
print(f"Z-statistic: {z_c:.4f}")
print(f"P-value: {p_c:.4f}")
print(f"95% CI for difference: [{ci_c_low:.2f}, {ci_c_high:.2f}] percentage points")
print("Decision at alpha = 0.05:", "Reject H0" if p_c < 0.05 else "Fail to reject H0")

Complaint rate A: 20/819 = 2.44%
Complaint rate B: 26/980 = 2.65%
Difference (B - A): 0.21 percentage points
Z-statistic: 0.2824
P-value: 0.7776
95% CI for difference: [-1.25, 1.67] percentage points
Decision at alpha = 0.05: Fail to reject H0


### 3.8 Guardrail 2: Return Rate (among purchasers)
- **H0:** Return rate of A = Return rate of B
- **H1:** Return rate of A ≠ Return rate of B
- **Denominator:** purchasers only (819 in A, 980 in B)
- **Test:** Two-proportion z-test, alpha = 0.05

In [21]:
# Returns and purchasers per group
r_A = int(df[df["Group"] == "A"]["Returned"].sum())
r_B = int(df[df["Group"] == "B"]["Returned"].sum())

rr_A = r_A / pur_A
rr_B = r_B / pur_B

# Z-test
z_r, p_r = proportions_ztest([r_B, r_A], [pur_B, pur_A], alternative="two-sided")

# 95% CI for the difference (B - A)
diff_r = rr_B - rr_A
se_r = np.sqrt(rr_A * (1 - rr_A) / pur_A + rr_B * (1 - rr_B) / pur_B)
ci_r_low = (diff_r - 1.96 * se_r) * 100
ci_r_high = (diff_r + 1.96 * se_r) * 100

print(f"Return rate A: {r_A}/{pur_A} = {rr_A*100:.2f}%")
print(f"Return rate B: {r_B}/{pur_B} = {rr_B*100:.2f}%")
print(f"Difference (B - A): {diff_r*100:.2f} percentage points")
print(f"Z-statistic: {z_r:.4f}")
print(f"P-value: {p_r:.4f}")
print(f"95% CI for difference: [{ci_r_low:.2f}, {ci_r_high:.2f}] percentage points")
print("Decision at alpha = 0.05:", "Reject H0" if p_r < 0.05 else "Fail to reject H0")

Return rate A: 45/819 = 5.49%
Return rate B: 52/980 = 5.31%
Difference (B - A): -0.19 percentage points
Z-statistic: -0.1762
P-value: 0.8602
95% CI for difference: [-2.29, 1.91] percentage points
Decision at alpha = 0.05: Fail to reject H0


### 3.9 AOV Sanity Check
Every completed order is designed to be Rs 800. This check confirms the order values contain no variation, so no significance test is needed for AOV.

In [22]:
purchasers_df = df[df["Purchased"] == 1]

aov_check = purchasers_df.groupby("Group")["Order_Value"].agg(
    orders="count",
    aov="mean",
    min_value="min",
    max_value="max",
    unique_values="nunique"
)

aov_check

,orders,aov,min_value,max_value,unique_values
Group,,,,,
A,819,800.0,800,800,1
B,980,800.0,800,800,1


### 3.10 Final Statistical Summary
All test results in one table. Differences are Group B minus Group A, in percentage points (pp).

In [23]:
final_summary = pd.DataFrame([
    {
        "Metric": "Purchase Conversion (primary)",
        "Group A": f"{p_A*100:.2f}%",
        "Group B": f"{p_B*100:.2f}%",
        "Difference (pp)": round(diff * 100, 2),
        "95% CI (pp)": f"[{ci_low:.2f}, {ci_high:.2f}]",
        "P-value": round(p_value, 6),
        "Significant (alpha=0.05)": "Yes" if p_value < 0.05 else "No",
    },
    {
        "Metric": "Complaint Rate (guardrail)",
        "Group A": f"{cr_A*100:.2f}%",
        "Group B": f"{cr_B*100:.2f}%",
        "Difference (pp)": round(diff_c * 100, 2),
        "95% CI (pp)": f"[{ci_c_low:.2f}, {ci_c_high:.2f}]",
        "P-value": round(p_c, 4),
        "Significant (alpha=0.05)": "Yes" if p_c < 0.05 else "No",
    },
    {
        "Metric": "Return Rate (guardrail)",
        "Group A": f"{rr_A*100:.2f}%",
        "Group B": f"{rr_B*100:.2f}%",
        "Difference (pp)": round(diff_r * 100, 2),
        "95% CI (pp)": f"[{ci_r_low:.2f}, {ci_r_high:.2f}]",
        "P-value": round(p_r, 4),
        "Significant (alpha=0.05)": "Yes" if p_r < 0.05 else "No",
    },
])

final_summary

,Metric,Group A,Group B,Difference (pp),95% CI (pp),P-value,Significant (alpha=0.05)
0,Purchase Conversion (primary),8.19%,9.80%,1.61,"[0.82, 2.40]",0.000069,Yes
1,Complaint Rate (guardrail),2.44%,2.65%,0.21,"[-1.25, 1.67]",0.777600,No
2,Return Rate (guardrail),5.49%,5.31%,-0.19,"[-2.29, 1.91]",0.860200,No


### Phase 3 Summary: Statistical Analysis
| Metric | Group A (₹200 OFF) | Group B (20% OFF) | Difference (pp) | 95% CI (pp) | P-value | Significant (α = 0.05) |
|---|---|---|---|---|---|---|
| Purchase Conversion (primary) | 8.19% | 9.80% | +1.61 | [0.82, 2.40] | 0.000069 | Yes |
| Complaint Rate (guardrail) | 2.44% | 2.65% | +0.21 | [-1.25, 1.67] | 0.7776 | No |
| Return Rate (guardrail) | 5.49% | 5.31% | -0.19 | [-2.29, 1.91] | 0.8602 | No |

- **Relative lift in conversion:** 19.66%
- **AOV:** Rs 800 in both groups (identical by design, no test needed)
- **Test used:** Two-proportion z-test, two-sided, α = 0.05
- **Estimated impact (illustrative):** about 161 extra purchases per 10,000 users (plausible range about 82 to 240)

**Conclusion:** In this synthetic dataset, "20% OFF" shows a statistically significant higher purchase conversion than "₹200 OFF". No statistical evidence of a change in complaint or return rates was found, although the guardrail intervals are wide because the counts are small.